# PKG Attrition v13 · Model 1 — one simple model on all features

**Payment Knowledge Graph (PKG) · Treasury Management · Data Science**

Sources: the daily deposit table and `pkg_cust_metrics_monthly`. Nothing else is read and no table is written;
everything lands in `OUT_DIR` (CSVs, charts, and the HTML report built in the last cell).

| § | What it does |
|---|---|
| 1 | Setup |
| 2 | Deposit data → monthly customer panel, targets, deposit features |
| 3 | Payment metrics → monthly check, power-id key, payment features (level, change, silent months) |
| 4 | Model frame: rows at risk, target, training sample, feature sets |
| 5 | Rolling monthly test folds: fit, score, accuracy by test month |
| 6 | Predictive power: single features and families |
| 7 | How much is at risk; accuracy and savings against calls, α = 0 / 0.5 / 1; calibration |
| 8 | When signals fire (event time, model view, queue time); leavers vs stayers |
| 9 | Example clients for RM managers |
| 10 | Prediction scorecard and the HTML report |

**The model is deliberately simple:** ridge logistic regression on standardised features, fitted by damped
Newton in numpy, with no dependence on sklearn or XGBoost. Every score splits exactly into per-feature
contributions, which gives the reason codes and the RM examples. A tree model is the next version.

**Outputs are consolidated:** each code cell prints one block. Charts go to disk and into the report, not inline.

### Pre-registered predictions — scored in §10

| # | Prediction | Confidence |
|---|---|---|
| P1 | Pooled test AUC on all at-risk client-months ≥ 0.80 | 60% |
| P2 | The family with the largest share of model contribution is a payment family, not the deposit block | 40% |
| P3 | A bank-count feature (`n_fi*`) is among the ten features with the largest contribution | 55% |
| P4 | At 1,000 calls and α = 0.5, the best test month's precision is at most 2× the worst month's | 50% |
| P5 | At 1,000 calls, α = 0.5 reaches more capped dollars than α = 0 | 80% |
| P6 | At 1,000 calls and α = 0.5, ≥ 30% of departing clients are first alerted 3+ months before the money leaves | 55% |
| P7 | At least 3 payment features fire (event time) at or before 6 months out | 50% |

In [ ]:
# =====================================================================
# 0 · CONFIG
# =====================================================================
DEP_TABLE  = "dsihd01p_dsi.lap_dsi_universe_optimized"          # daily: account × leg × business day
MET_TABLE  = "bdahd01p_dlcdi1_cdi_tm.pkg_cust_metrics_monthly"
DICT_TABLE = "bdahd01p_dlcdi1_cdi_tm.pkg_cust_metrics_dictionary"
OUT_DIR    = "../metrics/attrition/v13/m1"
DATE_START, DATE_END = "2024-01-01", "2026-08-31"                # nothing before 2024 is trusted
SHUFFLE_PARTITIONS = 800

# ── status codes: two vocabularies in one column (settled in v2) ─────
STATUS_CLASS = {"01": "open", "99": "open", "09": "open", "O": "open",
                "03": "inactive", "05": "dormant", "06": "escheat", "12": "closing",
                "07": "closed", "08": "closed", "C": "closed"}
LIVE_CLASSES = {"open", "inactive", "dormant", "escheat", "closing"}
PRICING = ["earnings_credit_rate", "curr_int_rate"]              # used if present

# ── definitions ──────────────────────────────────────────────────────
H          = 6          # horizon: does the money leave within the next 6 months?
CONFIRM_M  = 2          # a full exit is confirmed after 2 observed non-live months
NORM_WIN, NORM_MIN_OBS = 12, 3    # normal balance: median of up to 12 prior live months, needs 3
MIN_HIST   = 3          # months observed before a client-month can be scored
MSH_FRAC, MSH_FLOOR = 0.50, 10_000     # money still here: ≥ half of normal, normal ≥ $10k
BAL_EXIT_FRAC, BAL_EXIT_HOLD = 0.05, 3 # balance exit: < 5% of normal for 3 months, accounts open
CHURN_WARM = 6          # months of payment history before the churn-state metrics are used

# ── training ─────────────────────────────────────────────────────────
NEG_RATE   = 0.10       # share of non-leaving client-months kept for training (weight 1/NEG_RATE)
SEED       = 13
L2         = 2.0        # ridge penalty on standardised features (on the summed log-likelihood)
CLIP_Q     = (0.005, 0.995)
TEST_FIRST = "2025-04"  # first test month; the last is the last month with a fully observed outcome

# ── queue and value ──────────────────────────────────────────────────
CALLS      = [100, 250, 500, 1000, 2000]   # calls per month
ALPHAS     = [0.0, 0.5, 1.0]               # score = p × normal balance^α
SAVE_RATES = [0.01, 0.05, 0.10, 0.20]      # illustrative; no call cost anywhere
K_MAIN, A_MAIN = 1000, 0.5
BANDS      = [(0, 1e5, "< $100k"), (1e5, 1e6, "$100k–1m"), (1e6, 1e7, "$1m–10m"), (1e7, float("inf"), "$10m+")]

# ── signal timing and examples ───────────────────────────────────────
N_TIMING   = 30         # features traced month by month before exit
LIFT_BAR   = 1.5        # a signal "fires" when leavers cross the stayers' 10th/90th percentile ≥ 1.5× as often
EVENT_PRE  = 12
MAX_EVENT_CUST = 15_000 # per group, for the event-time pulls

In [ ]:
# =====================================================================
# 1 · SETUP — shims, session, helpers
# =====================================================================
import numpy as np, warnings
for _a, _t in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _a): setattr(np, _a, _t)
if "bool" not in np.__dict__: np.bool = np.bool_
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"): pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):    pd.Series.iteritems = pd.Series.items
for _c in (ResourceWarning, DeprecationWarning, FutureWarning, RuntimeWarning):
    warnings.filterwarnings("ignore", category=_c)

import os, re, time, json, base64, hashlib, pathlib, html as _html
from pyspark.sql import SparkSession, Window, functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")            # features are cast to float before collecting
spark.conf.set("spark.sql.execution.arrow.pyspark.fallback.enabled", "true")
DISK = StorageLevel.DISK_ONLY
OUT = pathlib.Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
T0 = time.time()
R = {}          # every number the report uses
CHARTS = {}     # chart name → file
CHECKS = []

# ── one consolidated output per cell ─────────────────────────────────
def _fmt_col(s, kind):
    if kind == "pct":   return s.map(lambda v: "—" if pd.isna(v) else f"{v:.1%}")
    if kind == "money": return s.map(lambda v: "—" if pd.isna(v) else f"${v:,.0f}")
    if pd.api.types.is_integer_dtype(s): return s.map(lambda v: f"{v:,}")
    if pd.api.types.is_float_dtype(s):   return s.map(lambda v: "—" if pd.isna(v) else f"{v:,.3f}")
    return s.map(lambda v: "—" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))

def fmt_df(df, pct=(), money=()):
    d = df.copy()
    for c in d.columns:
        d[c] = _fmt_col(d[c], "pct" if c in pct else "money" if c in money else None)
    return d

class Block:
    """Collects every table and note of a cell and displays them once, as one image-friendly output."""
    def __init__(self, title):
        self.h = [f"<div style='font-family:Arial,sans-serif;font-size:12px'><h3 style='margin:2px 0 6px'>{title}</h3>"]
    def table(self, df, title, pct=(), money=(), save=None, max_rows=80):
        if save: df.to_csv(OUT / f"{save}.csv", index=False)
        d = fmt_df(df.head(max_rows), pct, money)
        self.h.append(f"<p style='margin:8px 0 2px'><b>{title}</b> <span style='color:#777'>[{len(df):,} rows]</span></p>" +
                      d.to_html(index=False, border=0, classes="t").replace(
                          "<table", "<table style='border-collapse:collapse;font-size:11px' cellpadding='3'"))
        return self
    def kv(self, pairs, title):
        return self.table(pd.DataFrame([{"item": k, "value": v if isinstance(v, str) else
                          f"{v:,}" if isinstance(v, (int, np.integer)) else f"{v:,.4f}" if isinstance(v, float) else str(v)}
                                        for k, v in pairs]), title)
    def note(self, s):
        self.h.append(f"<p style='margin:4px 0;color:#333'>{s}</p>"); return self
    def check(self, item, status, detail):
        CHECKS.append({"item": item, "status": status, "detail": detail})
        col = {"READY": "#2e7d32", "FIX": "#e65100", "BLOCKER": "#b71c1c", "INFO": "#555"}[status]
        return self.note(f"<span style='color:{col};font-weight:bold'>[{status}]</span> {item}: {detail}")
    def show(self, t0=None):
        if t0: self.note(f"<span style='color:#999'>wall {time.time() - t0:,.0f}s</span>")
        display(HTML("".join(self.h) + "</div>"))

def savefig(fig, name):
    p = OUT / f"{name}.png"; fig.savefig(p, dpi=120, bbox_inches="tight"); plt.close(fig); CHARTS[name] = p

def topd(df):
    for c, t in df.dtypes:
        if t.startswith("decimal"): df = df.withColumn(c, F.col(c).cast("double"))
    return df.toPandas()

def nz(c):
    s = F.trim(F.col(c).cast("string")); return F.when(s == "", F.lit(None)).otherwise(s)

# ── months: offset t from the first month only ───────────────────────
def ym_idx(ym): y, m = map(int, ym[:7].split("-")); return y * 12 + m - 1
M0 = ym_idx(DATE_START)
def t_of(ym): return ym_idx(ym) - M0
def ym_of(t): i = M0 + int(t); return f"{i // 12:04d}-{i % 12 + 1:02d}"
def t_col(c): return (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int") - 1 - F.lit(M0)).cast("int")
assert t_of(DATE_START) == 0 and ym_of(0) == DATE_START[:7]

def cid(x):
    """Display id: one-way hash. Full ids are only ever written to *_FULL_IDS.csv."""
    return hashlib.sha1(str(x).encode()).hexdigest()[:10]

def auc(y, s):
    """Rank AUC with ties averaged; y in {0,1}."""
    y = np.asarray(y, float); s = np.asarray(s, float); ok = ~np.isnan(s); y, s = y[ok], s[ok]
    n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(s).rank(method="average").to_numpy()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

print(f"Spark {spark.version} | window {DATE_START} … {DATE_END} | outputs → {OUT.resolve()}")

## §2 Deposit data → monthly customer panel, targets, deposit features

Legs are summed per account-day; balances are rolled to customer-day, then month (average, minimum, maximum,
month-end). Every customer gets a complete month grid from its first month, so a month with no rows reads as
non-live with zero balance. All windows look backwards only.

**Target** (defined and argued in the report, §4): *the money leaves within the next 6 months*, the earlier of
a **full exit** (every account non-live, confirmed by 2 observed non-live months) and a **balance exit** (balance
below 5% of normal for 3 months while accounts stay open).

In [ ]:
# =====================================================================
# 2 · DEPOSIT PANEL, TARGETS, DEPOSIT FEATURES                  [OUTPUT BLOCK 2]
# =====================================================================
t0 = time.time(); B = Block("§2 · Deposit data → monthly customer panel")
raw = spark.table(DEP_TABLE)
REQ = ["acct_full_acct_id", "sub_product_cd", "edw_tda_load_dt", "balance", "acct_status", "deposit_family",
       "opened_dt", "closed_dt", "cust_pwr_id", "rltn_pwr_id"]
miss = [c for c in REQ if c not in raw.columns]
if miss: raise RuntimeError(f"deposit table lacks {miss}")
PRC = [c for c in PRICING if c in raw.columns]
TIEB = [c for c in ["bdh_hdfs_load_ts", "cod_hdfs_load_ts"] if c in raw.columns]
SMAP = F.create_map(*[F.lit(x) for kv_ in STATUS_CLASS.items() for x in kv_])
dep = raw.select(*REQ, *PRC, *TIEB)
for c in ["acct_full_acct_id", "sub_product_cd", "cust_pwr_id", "rltn_pwr_id", "acct_status", "deposit_family"]:
    dep = dep.withColumn(c, nz(c))
for c in ["edw_tda_load_dt", "opened_dt", "closed_dt"]: dep = dep.withColumn(c, F.to_date(F.col(c)))
for c in ["balance"] + PRC: dep = dep.withColumn(c, F.col(c).cast("double"))
dep = (dep.filter(F.col("edw_tda_load_dt").between(F.lit(DATE_START), F.lit(DATE_END)))
          .withColumn("leg", F.coalesce(F.col("sub_product_cd"), F.lit("__NA__")))
          .withColumn("ym", F.date_format("edw_tda_load_dt", "yyyy-MM"))
          .withColumn("status_class", F.coalesce(SMAP[F.col("acct_status")], F.lit("UNMAPPED")))
          .withColumn("live_leg", F.col("status_class").isin(*LIVE_CLASSES).cast("int")))

unm = [r[0] for r in dep.filter("status_class = 'UNMAPPED'").select("acct_status").distinct().limit(20).collect()]
if unm: raise RuntimeError(f"unmapped acct_status codes {unm}: add them to STATUS_CLASS")
n_dup = dep.groupBy("acct_full_acct_id", "leg", "edw_tda_load_dt").count().filter("count > 1").count()
if n_dup:
    _o = [F.col(c).desc_nulls_last() for c in TIEB] + [F.col("balance").desc_nulls_last()]
    dep = dep.withColumn("_rn", F.row_number().over(Window.partitionBy("acct_full_acct_id", "leg", "edw_tda_load_dt")
                                                    .orderBy(*_o))).filter("_rn = 1").drop("_rn")
MC = topd(dep.groupBy("ym").agg(F.countDistinct("edw_tda_load_dt").alias("bdays"),
                                F.countDistinct("cust_pwr_id").alias("customers")).orderBy("ym"))
END_M = MC.loc[MC.bdays >= 15, "ym"].max(); END_T = t_of(END_M)
holes = [ym_of(t) for t in range(END_T + 1) if ym_of(t) not in set(MC.loc[MC.bdays >= 15, "ym"])]
if holes: raise RuntimeError(f"deposit months missing or partial inside the window: {holes}")
dep = dep.withColumn("t", t_col("ym")).filter(F.col("t") <= END_T)

# ── customer-month balances (customer-day first, so daily min / max are real) ──
cday = (dep.filter("cust_pwr_id is not null").groupBy("cust_pwr_id", "edw_tda_load_dt", "t")
           .agg(F.sum("balance").alias("bal_d")))
cbal = cday.groupBy("cust_pwr_id", "t").agg(F.avg("bal_d").alias("bal_avg"), F.min("bal_d").alias("bal_min"),
                                            F.max("bal_d").alias("bal_max"),
                                            F.expr("max_by(bal_d, edw_tda_load_dt)").alias("bal_eom"))
# ── account-month → customer-month counts, product mix, pricing ─────
am = dep.groupBy("acct_full_acct_id", "t", "ym").agg(
    F.max("live_leg").alias("live"),
    F.expr("max_by(cust_pwr_id, edw_tda_load_dt)").alias("cust_pwr_id"),
    F.expr("max_by(rltn_pwr_id, edw_tda_load_dt)").alias("rltn_pwr_id"),
    F.expr("max_by(deposit_family, abs(coalesce(balance, 0)))").alias("deposit_family"),
    (F.sum("balance") / F.countDistinct("edw_tda_load_dt")).alias("acct_bal"),
    F.min("opened_dt").alias("opened_dt"),
    F.max((F.date_format("closed_dt", "yyyy-MM") == F.col("ym")).cast("int")).alias("closed_in_m"),
    *[F.expr(f"max_by({c}, edw_tda_load_dt)").alias(c) for c in PRC])
fam = F.upper(F.coalesce(F.col("deposit_family"), F.lit("")))
pos = F.greatest(F.col("acct_bal"), F.lit(0.0))
fb = lambda cond: F.sum(F.when(cond, pos).otherwise(0.0))
def wavg(c):
    ok = F.col(c).isNotNull() & (F.col("acct_bal") > 0)
    return (F.sum(F.when(ok, F.col(c) * F.col("acct_bal"))) / F.sum(F.when(ok, F.col("acct_bal")))).alias(c)
cacc = am.filter("cust_pwr_id is not null").groupBy("cust_pwr_id", "t").agg(
    F.count("*").alias("n_accts"), F.sum("live").alias("n_live"),
    F.sum((F.date_format("opened_dt", "yyyy-MM") == F.col("ym")).cast("int")).alias("n_opened"),
    F.sum("closed_in_m").alias("n_closed"),
    fb(fam == "IBDDA").alias("fb_ibdda"), fb(fam == "MMDA").alias("fb_mmda"), fb(fam.contains("SWEEP")).alias("fb_sweep"),
    fb(F.lit(True)).alias("fb_all"), F.max("rltn_pwr_id").alias("rltn_pwr_id"), *[wavg(c) for c in PRC])
cm = cbal.join(cacc, ["cust_pwr_id", "t"], "full")

# ── complete month grid ──────────────────────────────────────────────
first = cm.groupBy("cust_pwr_id").agg(F.min("t").alias("first_t"))
grid = first.crossJoin(F.broadcast(spark.range(0, END_T + 1).select(F.col("id").cast("int").alias("t")))) \
            .filter(F.col("t") >= F.col("first_t"))
X = grid.join(cm, ["cust_pwr_id", "t"], "left")
for c in ["n_accts", "n_live", "n_opened", "n_closed"]: X = X.withColumn(c, F.coalesce(F.col(c), F.lit(0)))
for c in ["bal_avg", "bal_min", "bal_max", "bal_eom", "fb_ibdda", "fb_mmda", "fb_sweep", "fb_all"]:
    X = X.withColumn(c, F.coalesce(F.col(c), F.lit(0.0)))
wc = Window.partitionBy("cust_pwr_id"); wo = wc.orderBy("t")
rg = lambda a, b: wo.rangeBetween(a, b)
X = (X.withColumn("rltn_pwr_id", F.last("rltn_pwr_id", ignorenulls=True).over(wo.rowsBetween(Window.unboundedPreceding, 0)))
      .withColumn("bal", F.greatest(F.col("bal_avg"), F.lit(0.0)))
      .withColumn("live", (F.col("n_live") > 0).cast("int"))
      .withColumn("hist_t", F.col("t") - F.col("first_t") + 1)
      .withColumn("last_live_t", F.max(F.when(F.col("live") == 1, F.col("t"))).over(wc))
      .withColumn("exit_t", F.when(F.col("last_live_t") < END_T, F.col("last_live_t") + 1).cast("int"))
      .withColumn("a_t", F.when(F.lit(END_T) - F.col("exit_t") + 1 >= CONFIRM_M, F.col("exit_t")))
      .withColumn("_bl", F.when(F.col("live") == 1, F.col("bal")))
      .withColumn("bal_norm", F.expr("percentile_approx(_bl, 0.5)").over(rg(-NORM_WIN, -1)))
      .withColumn("_nn", F.count("_bl").over(rg(-NORM_WIN, -1)))
      .withColumn("bal_norm", F.when(F.col("_nn") >= NORM_MIN_OBS, F.col("bal_norm")))
      .withColumn("msh", F.coalesce(((F.col("live") == 1) & (F.col("bal_norm") >= MSH_FLOOR)
                                     & (F.col("bal") >= MSH_FRAC * F.col("bal_norm"))).cast("int"), F.lit(0)))
      .withColumn("capped", F.least(F.col("bal"), F.col("bal_norm"))))
thr = F.lit(BAL_EXIT_FRAC) * F.col("bal_norm")
bc = (F.col("live") == 1) & (F.col("bal_norm") > 0) & (F.col("bal") < thr) & (F.col("t") + BAL_EXIT_HOLD - 1 <= END_T)
for k in range(1, BAL_EXIT_HOLD): bc = bc & (F.lead("bal", k).over(wo) < thr)
X = (X.withColumn("b_t", F.min(F.when(F.coalesce(bc, F.lit(False)), F.col("t"))).over(wc).cast("int"))
      .withColumn("ab_t", F.least("a_t", "b_t"))
      .withColumn("exit_type", F.when(F.col("ab_t").isNull(), F.lit(None))
                                .when(F.col("a_t") == F.col("ab_t"), F.lit("full exit")).otherwise(F.lit("balance exit"))))

# ── deposit features (backward-looking only) ─────────────────────────
def chg(col, a, b, min_obs=2):
    return F.when(F.count(col).over(rg(a, b)) >= min_obs, F.col(col) - F.avg(col).over(rg(a, b)))
X = X.withColumn("_lb", F.log1p("bal")).withColumn("_nl", F.col("n_live").cast("double"))
DEP = {}
def add(name, expr, group, label):
    global X; X = X.withColumn(name, expr); DEP[name] = (group, label)
add("dep_log_bal", F.col("_lb"), "balance level", "balance, log")
add("dep_bal_vs_norm", F.col("_lb") - F.log1p("bal_norm"), "balance level", "balance vs normal balance (log ratio)")
add("dep_d1", F.col("_lb") - F.lag("_lb", 1).over(wo), "balance change", "balance change vs last month")
add("dep_d3", chg("_lb", -3, -1), "balance change", "balance change vs the last 3 months")
add("dep_d6", chg("_lb", -6, -4), "balance change", "balance change vs 4–6 months ago")
add("dep_d12", chg("_lb", -12, -10), "balance change", "balance change vs 10–12 months ago")
add("dep_drawdown", F.col("_lb") - F.max("_lb").over(rg(-12, -1)), "balance change", "balance vs its 12-month peak")
add("dep_vol6", F.when(F.count("_lb").over(rg(-5, 0)) >= 3, F.stddev("_lb").over(rg(-5, 0))), "balance volatility",
    "volatility of balance over 6 months")
add("dep_range", F.log1p(F.greatest("bal_max", F.lit(0.0))) - F.log1p(F.greatest("bal_min", F.lit(0.0))),
    "balance volatility", "within-month range of daily balance")
add("dep_eom_vs_avg", F.log1p(F.greatest("bal_eom", F.lit(0.0))) - F.col("_lb"), "balance volatility",
    "month-end balance vs monthly average")
add("dep_n_accts", F.log1p("n_accts"), "accounts", "number of accounts")
add("dep_n_live", F.log1p("n_live"), "accounts", "number of live accounts")
add("dep_live_d6", chg("_nl", -6, -4), "accounts", "change in live accounts vs 4–6 months ago")
add("dep_opened_3m", F.sum("n_opened").over(rg(-2, 0)), "accounts", "accounts opened in the last 3 months")
add("dep_closed_3m", F.sum("n_closed").over(rg(-2, 0)), "accounts", "accounts closed in the last 3 months")
for c, lab in [("fb_ibdda", "interest-bearing DDA"), ("fb_mmda", "money market"), ("fb_sweep", "sweep")]:
    add(f"dep_share_{c[3:]}", F.when(F.col("fb_all") > 0, F.col(c) / F.col("fb_all")), "product mix", f"share of balance in {lab}")
add("dep_tenure", F.log1p("hist_t"), "tenure", "months observed (log)")
add("dep_at_start", (F.col("first_t") == 0).cast("int"), "tenure", "client already present when the data starts")
for c in PRC:
    add(f"dep_{c}", F.col(c), "pricing", c.replace("_", " "))
    add(f"dep_{c}_d6", chg(c, -6, -4), "pricing", f"change in {c.replace('_', ' ')} vs 4–6 months ago")
rl = X.filter("rltn_pwr_id is not null").groupBy("rltn_pwr_id", "t").agg(F.sum("bal").alias("rl_bal"),
                                                                         F.countDistinct("cust_pwr_id").alias("rl_n"))
X = X.join(rl, ["rltn_pwr_id", "t"], "left").withColumn(
    "_lo", F.when(F.col("rl_n") > 1, F.log1p(F.greatest(F.col("rl_bal") - F.col("bal"), F.lit(0.0)))))
add("dep_siblings", F.log1p(F.coalesce(F.col("rl_n"), F.lit(1)) - 1), "relationship", "other customers in the same relationship")
add("dep_sib_d6", chg("_lo", -6, -4), "relationship", "change in the relationship's other balances vs 4–6 months ago")
DEP_FEATS = list(DEP)
X = X.drop("_bl", "_nn", "_nl", "_lo").persist(DISK)

# ── summary ──────────────────────────────────────────────────────────
g = X.agg(F.count("*").alias("rows"), F.countDistinct("cust_pwr_id").alias("customers"),
          F.sum("live").alias("live_rows"), F.sum("msh").alias("msh_rows")).first().asDict()
lab = topd(X.groupBy("cust_pwr_id").agg(F.first("exit_type").alias("exit_type"), F.first("exit_t").alias("exit_t"),
                                        F.first("a_t").alias("a_t"), F.first("b_t").alias("b_t"))
            .groupBy("exit_type").agg(F.count("*").alias("customers"),
                                      F.sum(F.col("exit_t").isNotNull().cast("int")).alias("also_full_exit_or_provisional"))
            .orderBy("exit_type"))
# is a full exit absorbing?  P(client comes back | ≥ CONFIRM_M observed non-live months)
nl = X.filter((F.col("live") == 0) & (F.col("t") > F.col("first_t")))
nl = nl.withColumn("_g", F.col("t") - F.row_number().over(Window.partitionBy("cust_pwr_id").orderBy("t")))
runs = nl.groupBy("cust_pwr_id", "_g").agg(F.count("*").alias("len"), F.min("t").alias("s"), F.first("last_live_t").alias("llt"))
rr = runs.filter(F.col("len") >= CONFIRM_M).agg(F.count("*").alias("n"), F.sum((F.col("s") < F.col("llt")).cast("int")).alias("back")).first()
R.update(dep_end=END_M, customers=g["customers"], p_return=(rr["back"] or 0) / max(rr["n"], 1), n_dup_keys=n_dup,
         dep_feats=len(DEP_FEATS), pricing_cols=PRC)
B.kv([("deposit window (full months)", f"{DATE_START[:7]} … {END_M} ({END_T + 1} months)"),
      ("duplicated account × leg × day keys (removed)", n_dup), ("customer-months (complete grid)", g["rows"]),
      ("customers", g["customers"]), ("live customer-months", g["live_rows"]),
      ("live months with money still here", f"{g['msh_rows'] / g['live_rows']:.1%}"),
      (f"P(client returns | ≥ {CONFIRM_M} non-live months)", f"{R['p_return']:.2%}  (runs: {rr['n']:,})"),
      ("deposit features", f"{len(DEP_FEATS)} ({', '.join(sorted(set(v[0] for v in DEP.values())))})")], "2a · Panel")
lab["exit_type"] = lab.exit_type.fillna("no exit in window (or not yet confirmed)")
B.table(lab, "2b · Customers by target event (whole window, before any qualification)", save="m1_2b_events")
B.check("deposit panel", "READY", f"{END_M} is the last full month; exits are absorbing ({R['p_return']:.2%} return)"
        if R["p_return"] <= 0.02 else f"{R['p_return']:.2%} of confirmed exits come back — raise CONFIRM_M")
B.show(t0)

## §3 Payment metrics → payment features

Every numeric column of `pkg_cust_metrics_monthly` except the identity columns becomes three features:

| Feature | Meaning |
|---|---|
| `m` | the level: log1p for dollars and counts (signed for net flows), as stored for shares and ratios |
| `m__d6` | change vs the client's own months −6…−4 (a log ratio for dollars and counts, a difference for shares) |
| `m__d3` | change vs the client's own last 3 months |

A client with several `mdm_id`s under one power id takes the row of its largest entity that month. A month with
no row after the client's first row is **zero activity**: dollars and counts 0, shares empty. Churn-state metrics
(new / reactivated / retained / lost, recurring, retention) are left empty for the first 6 months of 2024, while
their 12-month lookback is too short to mean anything.

In [ ]:
# =====================================================================
# 3 · PAYMENT METRICS → PAYMENT FEATURES                         [OUTPUT BLOCK 3]
# =====================================================================
t0 = time.time(); B = Block("§3 · Payment metrics → payment features")
MET = spark.table(MET_TABLE).filter(F.col("month").between(DATE_START[:7], END_M))
ID_KNOWN = {"mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type", "naics_cd", "naics_desc",
            "naics2", "naics_status", "is_tm", "pwr_shared", "family_size", "customer_start_dt", "in_neo4j_customer",
            "in_cust_dim", "month", "history_months"}
try:
    DICT = topd(spark.table(DICT_TABLE))
    ID_KNOWN |= set(DICT.loc[DICT.family == "identity", "column"])
    FAMILY = dict(zip(DICT.column, DICT.family)); DEFN = dict(zip(DICT.column, DICT.definition))
except Exception as e:
    FAMILY, DEFN = {}, {}
    B.check("dictionary", "INFO", f"{DICT_TABLE} unreadable ({type(e).__name__}); families default to 'other'")
NUM = ("int", "bigint", "double", "float", "smallint", "tinyint", "long")
MCOLS = [c for c, t in MET.dtypes if c not in ID_KNOWN and (t in NUM or t.startswith("decimal"))]

# ── every month present and populated? ───────────────────────────────
nn = topd(MET.groupBy("month").agg(F.count("*").alias("rows"), *[F.count(c).alias(c) for c in MCOLS]).orderBy("month"))
want = [ym_of(t) for t in range(END_T + 1)]
absent = [m for m in want if m not in set(nn.month)]
pop = (nn[MCOLS] > 0)
usual = pop.mean() >= 0.5                                         # columns populated in most months
CHURN_RE = re.compile(r"^(new_|reactivated_|retained_|lost_|rec_|fi_rec_|retention|fi_retention)")
nt = nn.month.map(t_of).to_numpy()
empty = ~pop.loc[:, usual.values]
lookback = np.array([bool(CHURN_RE.match(c)) for c in empty.columns])
# lookback-based columns are legitimately empty while their history is short (first months of 2024)
bad = empty.loc[:, ~lookback].any(axis=1).to_numpy() & (nt >= 1) | empty.any(axis=1).to_numpy() & (nt >= CHURN_WARM)
stale = nn.loc[bad, "month"].tolist()
B.table(nn[["month", "rows"]].assign(empty_columns=empty.sum(axis=1).values),
        "3a · Metrics by month (empty_columns = columns populated in most months but empty here)", save="m1_3a_months")
if absent or stale:
    B.check("metrics months", "BLOCKER", f"absent {absent}, empty columns in {stale}")
    raise RuntimeError(f"metrics months not ready: absent {absent}, stale {stale}")
B.check("metrics months", "READY", f"all {len(want)} months present, {len(MCOLS)} metric columns populated throughout")

# ── key: deposit cust_pwr_id ↔ metrics customer_pwr_id (format decided from the data) ──
def pk_expr(c, mode):
    s = F.trim(F.col(c).cast("string")); s = F.when(s == "", F.lit(None)).otherwise(s)
    return F.regexp_replace(s, "^0+(?=.)", "") if mode == "strip0" else s
dk = X.select("cust_pwr_id").distinct()
mk = MET.select("customer_pwr_id").distinct()
rates = {m_: dk.select(pk_expr("cust_pwr_id", m_).alias("k")).join(
            mk.select(pk_expr("customer_pwr_id", m_).alias("k")).distinct().withColumn("h", F.lit(1)), "k", "left")
            .agg(F.avg(F.coalesce("h", F.lit(0)).cast("double"))).first()[0] for m_ in ("raw", "strip0")}
KEY = "strip0" if rates["strip0"] > rates["raw"] + 0.01 else "raw"
R["key_match"] = rates[KEY]
if R["key_match"] < 0.3: raise RuntimeError(f"power-id join broken: {rates}")

# ── one row per power id and month: the largest entity ───────────────
flow = sum(F.coalesce(F.col(c).cast("double"), F.lit(0.0)) for c in ["amt_all_in", "amt_all_out"] if c in MCOLS)
ACT = [c for c in ["ntxn_all_out", "ntxn_all_in", "ntxn_self_out", "ntxn_self_in"] if c in MCOLS]
pm = (MET.select(pk_expr("customer_pwr_id", KEY).alias("pwr"), t_col("month").alias("t"), "mdm_id",
                 *[F.col(c).cast("double").alias(c) for c in MCOLS])
         .filter("pwr is not null")
         .withColumn("_f", flow)
         .withColumn("_r", F.row_number().over(Window.partitionBy("pwr", "t").orderBy(F.desc("_f"), "mdm_id")))
         .withColumn("pay_n_mdm", F.count("*").over(Window.partitionBy("pwr", "t")))
         .filter("_r = 1").drop("_r", "_f", "mdm_id")
         .withColumn("_row", F.lit(1))
         .withColumn("_act", (sum(F.coalesce(F.col(c), F.lit(0.0)) for c in ACT) > 0).cast("int") if ACT else F.lit(1)))

def kind(c):
    if re.match(r"^(amt_|net_flow|avg_ticket|selfpay_amt)", c): return "money"
    if re.search(r"(share|hhi|retention|entropy|coverage|ratio)", c): return "bounded"
    return "count"
KIND = {c: kind(c) for c in MCOLS}

Y = X.withColumn("pwr", pk_expr("cust_pwr_id", KEY)).join(pm, ["pwr", "t"], "left")
wp = Window.partitionBy("cust_pwr_id").orderBy("t")
Y = (Y.withColumn("pay_first_t", F.min(F.when(F.col("_row") == 1, F.col("t"))).over(
            wp.rowsBetween(Window.unboundedPreceding, 0)))                    # first row known by month t
      .withColumn("pay_seen", F.col("pay_first_t").isNotNull().cast("int"))
      .withColumn("pay_silent", ((F.col("pay_seen") == 1) & F.col("_row").isNull()).cast("int"))
      .withColumn("_last_act", F.max(F.when(F.col("_act") == 1, F.col("t"))).over(wp.rowsBetween(Window.unboundedPreceding, 0)))
      .withColumn("pay_months_since_act", F.col("t") - F.col("_last_act"))
      .withColumn("pay_n_mdm", F.when(F.col("pay_seen") == 1, F.coalesce("pay_n_mdm", F.lit(0)))))
lv = []
for c in MCOLS:
    v = F.col(c)
    if KIND[c] == "money": e = F.signum(v) * F.log1p(F.abs(v))
    elif KIND[c] == "count": e = F.log1p(F.greatest(v, F.lit(0.0)))
    else: e = v
    if KIND[c] != "bounded": e = F.when(F.col("_row").isNull() & (F.col("pay_seen") == 1), F.lit(0.0)).otherwise(e)
    if CHURN_RE.match(c): e = F.when(F.col("t") < CHURN_WARM, F.lit(None)).otherwise(e)
    lv.append(e.alias(c))
Y = Y.select(*[c for c in Y.columns if c not in MCOLS], *lv)
wr = lambda a, b: wp.rangeBetween(a, b)
ch = []
for c in MCOLS:
    for nm, (a, b) in (("d6", (-6, -4)), ("d3", (-3, -1))):
        ch.append(F.when(F.count(c).over(wr(a, b)) >= 2, F.col(c) - F.avg(c).over(wr(a, b))).alias(f"{c}__{nm}"))
FEAT = Y.select("*", *ch).persist(DISK)
PAY_FEATS = [f for c in MCOLS for f in (c, f"{c}__d6", f"{c}__d3")]
STATUS_FEATS = ["pay_seen", "pay_silent", "pay_months_since_act", "pay_n_mdm"]

cov = FEAT.filter("live = 1").agg(F.count("*").alias("n"), F.avg("pay_seen").alias("seen"), F.avg("pay_silent").alias("silent"),
                                  F.sum(F.when(F.col("pay_seen") == 1, F.col("bal"))).alias("bal_seen"),
                                  F.sum("bal").alias("bal_all")).first()
R.update(pay_seen=cov["seen"], pay_seen_usd=cov["bal_seen"] / cov["bal_all"], pay_silent=cov["silent"],
         n_metrics=len(MCOLS), pay_feats=len(PAY_FEATS), key_mode=KEY)
fam_tab = pd.Series({c: FAMILY.get(c, "other") for c in MCOLS}).value_counts().rename_axis("family").reset_index(name="metrics")
B.kv([("power-id key match (deposit customers found in the metrics)", f"{R['key_match']:.1%} (mode {KEY})"),
      ("live customer-months with payment history", f"{cov['seen']:.1%} of months, {R['pay_seen_usd']:.1%} of balance"),
      ("…of which silent (no metrics row that month)", f"{cov['silent']:.1%} of live months"),
      ("metric columns → features", f"{len(MCOLS)} → {len(PAY_FEATS)} (+ {len(STATUS_FEATS)} activity-status features)")],
     "3b · Payment features")
B.table(fam_tab, "3c · Metric columns by dictionary family")
B.show(t0)

## §4 Model frame, training sample, feature sets

**Rows at risk:** live, observed ≥ 3 months, a normal balance exists, and the money has not yet left.
**Label:** the money leaves (full or balance exit) in the next 6 months. A month is labelled only if all six
following months can be confirmed, so the last labelled month is 8 months before the end of the data.
**Training sample:** every leaving client-month, plus 10% of the others drawn by a fixed hash, weighted 10× so
the fitted probabilities stay on the population scale.

In [ ]:
# =====================================================================
# 4 · MODEL FRAME, TRAINING SAMPLE, FEATURE SETS                  [OUTPUT BLOCK 4]
# =====================================================================
t0 = time.time(); B = Block("§4 · Model frame and feature sets")
LAST_LABEL_T = min(END_T - CONFIRM_M + 1, END_T - BAL_EXIT_HOLD + 1) - H
FEATS = DEP_FEATS + PAY_FEATS + STATUS_FEATS
MF = (FEAT.filter((F.col("live") == 1) & (F.col("hist_t") >= MIN_HIST) & F.col("bal_norm").isNotNull()
                  & (F.col("ab_t").isNull() | (F.col("t") < F.col("ab_t"))))
          .withColumn("y", F.coalesce(((F.col("ab_t") > F.col("t")) & (F.col("ab_t") <= F.col("t") + H)).cast("int"), F.lit(0)))
          .withColumn("labelled", (F.col("t") <= LAST_LABEL_T).cast("int"))
          .persist(DISK))
META = ["cust_pwr_id", "t", "y", "msh", "bal", "bal_norm", "capped", "ab_t", "exit_type"]
keep = F.expr(f"pmod(hash(cust_pwr_id, t, {SEED}), 10000)") < int(NEG_RATE * 10000)
TRs = (MF.filter("labelled = 1").filter((F.col("y") == 1) | keep)
         .withColumn("w", F.when(F.col("y") == 1, F.lit(1.0)).otherwise(F.lit(1.0 / NEG_RATE)))
         .select(*META, "w", *[F.col(f).cast("float").alias(f) for f in FEATS]))
TR = topd(TRs)
TR[FEATS] = TR[FEATS].astype("float32")
by_t = topd(MF.groupBy("t").agg(F.count("*").alias("at_risk"), F.sum("y").alias("leaving_6m"), F.sum("msh").alias("money_still_here"))
            .orderBy("t"))
by_t["month"] = by_t.t.map(ym_of); by_t["rate_6m"] = by_t.leaving_6m / by_t.at_risk
by_t["labelled"] = by_t.t <= LAST_LABEL_T
R.update(last_label=ym_of(LAST_LABEL_T), train_rows=len(TR), train_pos=int(TR.y.sum()),
         at_risk_rows=int(by_t.loc[by_t.labelled, "at_risk"].sum()),
         base_rate=float(by_t.loc[by_t.labelled, "leaving_6m"].sum() / by_t.loc[by_t.labelled, "at_risk"].sum()))

# ── missing-value flags: one per distinct missing pattern (identical patterns share a flag) ──
miss = TR[FEATS].isna().mean()
cand = [f for f in FEATS if 0.005 < miss[f] < 0.995]
sub = TR[cand].isna().to_numpy()[:min(len(TR), 200_000)]
groups = {}
for j, f in enumerate(cand):
    groups.setdefault(hashlib.md5(np.packbits(sub[:, j]).tobytes()).hexdigest(), []).append(f)
FLAG_REP = [v[0] for v in groups.values()]
FLAG_MEMBERS = {f"miss·{v[0]}": v for v in groups.values()}
DROP = [f for f in FEATS if miss[f] >= 0.995 or TR[f].nunique(dropna=True) <= 1]
FEATS = [f for f in FEATS if f not in DROP]

# ── feature sets: deposit subsections + dictionary families ─────────
def fam_of(f):
    if f in DEP: return "deposit · " + DEP[f][0]
    if f in STATUS_FEATS: return "payment · activity status"
    if f.startswith("miss·"): return "missing-value flags"
    base = f.split("__")[0]
    return "payment · " + FAMILY.get(base, "other")
def label(f):
    if f in DEP: return DEP[f][1]
    if f.startswith("miss·"): return f"not observed: {label(f[5:])}"
    st = {"pay_seen": "has payment history", "pay_silent": "no payments this month (silent)",
          "pay_months_since_act": "months since the last payment", "pay_n_mdm": "number of payment entities"}
    if f in st: return st[f]
    base, _, tr = f.partition("__")
    d = DEFN.get(base, base.replace("_", " "))
    return d + {"d6": " — change vs 4–6 months ago", "d3": " — change vs the last 3 months", "": ""}[tr]
FS = pd.DataFrame([{"feature": f, "set": fam_of(f), "coverage": 1 - miss[f]} for f in FEATS])
FST = FS.groupby("set").agg(features=("feature", "size"), median_coverage=("coverage", "median")).reset_index()
FST.loc[len(FST)] = ["missing-value flags", len(FLAG_REP), 1.0]
R.update(n_feats=len(FEATS), n_flags=len(FLAG_REP), feature_sets=FST, dropped=DROP)
B.kv([("labelled months", f"{ym_of(0)} … {ym_of(LAST_LABEL_T)} (outcome fully observed)"),
      ("at-risk client-months, labelled", R["at_risk_rows"]), ("leave within 6 months (base rate)", f"{R['base_rate']:.2%}"),
      ("training sample", f"{len(TR):,} rows · {R['train_pos']:,} leaving · stayers at {NEG_RATE:.0%}, weight {1 / NEG_RATE:.0f}"),
      ("features", f"{len(FEATS)} + {len(FLAG_REP)} missing-pattern flags (dropped as empty/constant: {len(DROP)})")],
     "4a · Model frame")
B.table(by_t[["month", "at_risk", "leaving_6m", "rate_6m", "money_still_here", "labelled"]], "4b · Rows at risk by month",
        pct=("rate_6m",), save="m1_4b_by_month")
B.table(FST, "4c · Feature sets", pct=("median_coverage",), save="m1_4c_feature_sets")
FS.assign(label=FS.feature.map(label)).to_csv(OUT / "m1_4c_feature_list.csv", index=False)
B.show(t0)

## §5 Rolling monthly test folds

For each test month T the model is fitted only on months whose 6-month outcome was already known at T
(origins ≤ T − 6), then scores every client at risk in T. Nothing about T or later is used. The fit is ridge
logistic regression on standardised, clipped features with missing-pattern flags; damped Newton, so every step
improves the objective.

In [ ]:
# =====================================================================
# 5 · ROLLING TEST FOLDS — fit, score, accuracy by month         [OUTPUT BLOCK 5]
# =====================================================================
t0 = time.time(); B = Block("§5 · Rolling monthly test folds")
TEST_T = list(range(t_of(TEST_FIRST), LAST_LABEL_T + 1))
if len(TEST_T) < 3: raise RuntimeError(f"only {len(TEST_T)} test months")

def prep_fit(Xtr):
    s = Xtr[:min(len(Xtr), 200_000)]
    lo, hi = np.nanquantile(s, CLIP_Q[0], axis=0), np.nanquantile(s, CLIP_Q[1], axis=0)
    xc = np.clip(Xtr, lo, hi)
    mu = np.nanmean(xc, axis=0); sd = np.nanstd(xc, axis=0); sd[~(sd > 1e-9)] = 1.0
    mu = np.nan_to_num(mu); lo = np.nan_to_num(lo); hi = np.nan_to_num(hi)
    fm = np.isnan(Xtr[:, FI_FLAG]).mean(axis=0)
    return dict(lo=lo, hi=hi, mu=mu, sd=sd, fm=fm)

def transform(Xr, P):
    z = (np.clip(Xr, P["lo"], P["hi"]) - P["mu"]) / P["sd"]
    z = np.nan_to_num(z, nan=0.0)
    fl = np.isnan(Xr[:, FI_FLAG]).astype(np.float32) - P["fm"]
    return np.hstack([z, fl]).astype(np.float32)

def logit_newton(Z, y, w, l2=L2, max_it=60, tol=1e-6):
    """Weighted ridge logistic by DAMPED Newton: each step is halved until the penalised
    log-likelihood improves. Intercept unpenalised. Chunked so the float32 design is never copied whole."""
    n, k = Z.shape; w = w / w.mean()
    pbar = np.average(y, weights=w); b = np.zeros(k + 1); b[0] = np.log(pbar / (1 - pbar))
    def eta(b): return b[0] + np.concatenate([Z[i:i + 50_000].astype(np.float64) @ b[1:] for i in range(0, n, 50_000)])
    def obj(e, b): return float(np.sum(w * (np.logaddexp(0, e) - y * e)) + 0.5 * l2 * b[1:] @ b[1:])
    e = eta(b); f = obj(e, b); conv = False
    for it in range(1, max_it + 1):
        p = 1 / (1 + np.exp(-np.clip(e, -35, 35))); r = w * (p - y); v = w * p * (1 - p)
        g = np.empty(k + 1); g[0] = r.sum(); Hs = np.zeros((k + 1, k + 1)); Hs[0, 0] = v.sum(); g[1:] = l2 * b[1:]
        for i in range(0, n, 50_000):
            Zc = Z[i:i + 50_000].astype(np.float64); rc, vc = r[i:i + 50_000], v[i:i + 50_000]
            g[1:] += Zc.T @ rc; Hs[1:, 0] += Zc.T @ vc; Hs[1:, 1:] += (Zc * vc[:, None]).T @ Zc
        Hs[0, 1:] = Hs[1:, 0]; Hs[1:, 1:] += l2 * np.eye(k)
        step = np.linalg.solve(Hs, g); s_ = 1.0
        while True:
            bn = b - s_ * step; en = eta(bn); fn = obj(en, bn)
            if fn <= f or s_ < 1e-8: break
            s_ *= 0.5
        b, e, f_old, f = bn, en, f, fn
        if np.max(np.abs(g)) / n < tol or abs(f_old - f) < 1e-10 * max(1.0, abs(f)): conv = True; break
    return b, dict(converged=conv, iters=it)

FI_FLAG = [FEATS.index(f) for f in FLAG_REP if f in FEATS]
FLAG_REP = [FEATS[j] for j in FI_FLAG]
COLS = FEATS + [f"miss·{f}" for f in FLAG_REP]
LABEL = {c: label(c) for c in COLS}; FAMOF = {c: fam_of(c) for c in COLS}
Xall = TR[FEATS].to_numpy(np.float32)
TR = TR.drop(columns=[c for c in TR.columns if c not in META + ["w"]])    # keep one copy of the training features
SCORES, FOLDS, ABS_SUM, N_SCORED, SAMPLE = [], [], np.zeros(len(COLS)), 0, []
rng = np.random.default_rng(SEED)
for T in TEST_T:
    m = (TR.t <= T - H).to_numpy()
    P = prep_fit(Xall[m]); Z = transform(Xall[m], P)
    b, info = logit_newton(Z, TR.y.to_numpy(float)[m], TR.w.to_numpy(float)[m])
    te = topd(MF.filter(F.col("t") == T).select(*META, *[F.col(f).cast("float").alias(f) for f in FEATS]))
    Xt = te[FEATS].to_numpy(np.float32); Zt = transform(Xt, P)
    C = Zt * b[1:].astype(np.float32)
    p = 1 / (1 + np.exp(-np.clip(b[0] + C.sum(axis=1), -35, 35)))
    top = np.argsort(-C, axis=1)[:, :3]
    te["p"] = p; te["tm"] = T
    for k_ in range(3):
        te[f"r{k_ + 1}"] = [COLS[j] for j in top[:, k_]]
        te[f"c{k_ + 1}"] = C[np.arange(len(C)), top[:, k_]]
    ABS_SUM += np.abs(C).sum(axis=0); N_SCORED += len(C)
    si = rng.random(len(te)) < min(1.0, 15_000 / max(len(te), 1))
    SAMPLE.append(pd.DataFrame(Xt[si], columns=FEATS).assign(y=te.y.values[si]))
    SCORES.append(te.drop(columns=FEATS))
    FOLDS.append({"test_month": ym_of(T), "train_months": f"{ym_of(int(TR.t.min()))}…{ym_of(T - H)}", "train_rows": int(m.sum()),
                  "train_leaving": int(TR.y.to_numpy()[m].sum()), "converged": info["converged"], "iters": info["iters"],
                  "test_rows": len(te), "test_leaving": int(te.y.sum()),
                  "auc_all": auc(te.y, p), "auc_money_here": auc(te.y[te.msh == 1], p[te.msh.to_numpy() == 1]),
                  "mean_p": float(p.mean()), "actual_rate": float(te.y.mean())})
    MODEL = dict(b=b, P=P)                                 # the last fold's model is kept for §8
    print(f"  {ym_of(T)}: {len(te):,} scored, AUC {FOLDS[-1]['auc_all']:.3f}, {info}")
SC = pd.concat(SCORES, ignore_index=True); SMP = pd.concat(SAMPLE, ignore_index=True)
FO = pd.DataFrame(FOLDS)
SC["capped"] = SC.capped.fillna(0.0).clip(lower=0)
R.update(test_months=[ym_of(t) for t in TEST_T], n_test=len(TEST_T), folds=FO,
         auc_pooled=auc(SC.y, SC.p), auc_pooled_msh=auc(SC.y[SC.msh == 1], SC.p[SC.msh == 1]),
         all_converged=bool(FO.converged.all()))
B.table(FO, "5a · One fold per test month (fitted only on outcomes known before the test month)",
        pct=("mean_p", "actual_rate"), save="m1_5a_folds")
B.kv([("pooled AUC, all client-months at risk", R["auc_pooled"]), ("pooled AUC, money still here", R["auc_pooled_msh"]),
      ("AUC range across test months", f"{FO.auc_all.min():.3f} – {FO.auc_all.max():.3f}"),
      ("every fit converged", str(R["all_converged"]))], "5b · Discrimination")
B.check("fits", "READY" if R["all_converged"] else "FIX",
        "all folds converged" if R["all_converged"] else "a fold did not converge: raise L2")
B.show(t0)

## §6 Predictive power

Two views, because they answer different questions. **Alone:** how well one feature ranks leavers by itself
(AUC folded so 0.5 = nothing, 1 = perfect; direction says whether leavers sit high or low). **In the model:** the
feature's average absolute contribution to the score on the test months, as a share of all contributions. A
feature can be strong alone and small in the model when another feature carries the same information.

In [ ]:
# =====================================================================
# 6 · PREDICTIVE POWER — single features and families             [OUTPUT BLOCK 6]
# =====================================================================
t0 = time.time(); B = Block("§6 · Predictive power")
share = ABS_SUM / ABS_SUM.sum()
b_last = MODEL["b"][1:]
alone = {}
for f in FEATS:
    a = auc(SMP.y, SMP[f].to_numpy(float))
    alone[f] = (max(a, 1 - a) if pd.notna(a) else np.nan, "higher" if (pd.notna(a) and a >= 0.5) else "lower",
                float(SMP[f].notna().mean()))
IMP = pd.DataFrame([{"feature": c, "label": LABEL[c], "set": FAMOF[c], "model_share": share[j],
                     "direction_in_model": "raises risk when high" if b_last[j] > 0 else "raises risk when low",
                     "alone_auc": alone.get(c, (np.nan,))[0], "leavers_sit": alone.get(c, (None, None))[1],
                     "coverage": alone.get(c, (None, None, np.nan))[2]} for j, c in enumerate(COLS)])
IMP = IMP.sort_values("model_share", ascending=False).reset_index(drop=True)
IMP.to_csv(OUT / "m1_6_importance_all.csv", index=False)
FAMI = (IMP.groupby("set").agg(model_share=("model_share", "sum"), features=("feature", "size"),
                               best_alone=("alone_auc", "max")).sort_values("model_share", ascending=False).reset_index())
ALONE = IMP.dropna(subset=["alone_auc"]).sort_values("alone_auc", ascending=False).head(15)
R.update(imp=IMP, fam_imp=FAMI, alone=ALONE)
B.table(IMP.head(25), "6a · Top 25 features by share of model contribution", pct=("model_share", "coverage"), save="m1_6a_top")
B.table(FAMI, "6b · Feature sets by share of model contribution", pct=("model_share",), save="m1_6b_sets")
B.table(ALONE[["feature", "label", "set", "alone_auc", "leavers_sit", "coverage"]], "6c · Strongest features on their own",
        pct=("coverage",))
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
t20 = IMP.head(20)[::-1]
ax[0].barh([l[:55] for l in t20.label], t20.model_share, color="#1f4e79"); ax[0].set_title("Top 20 features, share of contribution")
f10 = FAMI.head(12)[::-1]
ax[1].barh(f10.set, f10.model_share, color="#c55a11"); ax[1].set_title("Feature sets, share of contribution")
for a in ax: a.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
savefig(fig, "m1_6_importance")
B.show(t0)

## §7 How much is at risk; accuracy and savings against calls

**The list:** each test month, clients whose money is still here are ranked by `p × normal balance^α` and the
top K are called. α = 0 ranks by probability alone, α = 1 by expected dollars, α = 0.5 in between.
**Credit for a save** is the capped balance at the alert (the lesser of current and normal balance), so a one-off
inflow is never counted. **Dollars reached** count each departing client once, at its first alert; the total over
the test months is annualised. **Savings** = dollars reached × an assumed save rate. There is no call cost.

In [ ]:
# =====================================================================
# 7 · AT RISK · ACCURACY VS CALLS · SAVINGS · CALIBRATION          [OUTPUT BLOCK 7]
# =====================================================================
t0 = time.time(); B = Block("§7 · At risk, accuracy, savings")
# ── 7a · how much is at risk (every confirmed exit in the labelled window) ──
ex = topd(FEAT.filter(F.col("ab_t").isNotNull() & (F.col("ab_t") <= LAST_LABEL_T + H) & (F.col("ab_t") >= MIN_HIST + 1))
              .groupBy("cust_pwr_id").agg(F.first("ab_t").alias("ab_t"), F.first("exit_type").alias("exit_type"),
                                          F.max(F.when(F.col("t") == F.col("ab_t") - 1, F.col("bal_norm"))).alias("norm"),
                                          F.max(F.when(F.col("t") == F.col("ab_t") - 3, F.col("msh"))).alias("msh_m3"),
                                          F.max(F.when(F.col("t") == F.col("ab_t") - 3, F.col("capped"))).alias("capped_m3")))
ex = ex.dropna(subset=["norm"])
span = (LAST_LABEL_T + H) - (MIN_HIST + 1) + 1
ex["band"] = pd.cut(ex.norm, [b[0] for b in BANDS] + [BANDS[-1][1]], labels=[b[2] for b in BANDS], right=False)
AR = (ex.groupby(["band", "exit_type"], observed=False).agg(clients=("norm", "size"), normal_balance=("norm", "sum"))
        .reset_index())
AR["per_year"] = AR.normal_balance * 12 / span
tot = ex.norm.sum() * 12 / span
R.update(at_risk_year=tot, exits_per_year=len(ex) * 12 / span, at_risk_tab=AR,
         msh_m3=float(ex.msh_m3.mean()), capped_m3_year=float(ex.capped_m3.fillna(0).sum() * 12 / span),
         top_client_share=float(ex.norm.max() / ex.norm.sum()) if len(ex) else np.nan)
B.kv([("clients whose money left (per year)", f"{R['exits_per_year']:,.0f}"),
      ("their normal balance (per year)", f"${tot:,.0f}"),
      ("…still here 3 months before the exit (clients)", f"{R['msh_m3']:.1%}"),
      ("…capped balance still defendable 3 months before (per year)", f"${R['capped_m3_year']:,.0f}"),
      ("largest single client's share of the total", f"{R['top_client_share']:.1%}")], "7a · How much is at risk")
B.table(AR, "7a · By size band and exit type (annualised)", money=("normal_balance", "per_year"), save="m1_7a_at_risk")

# ── 7b · the list: per month and pooled ──────────────────────────────
Q = SC[SC.msh == 1].copy()
Q["norm"] = Q.bal_norm.clip(lower=1.0)
per_month, first_alert = [], {}
for a in ALPHAS:
    Q[f"s{a}"] = Q.p * Q.norm ** a
    Q[f"rank{a}"] = Q.groupby("tm")[f"s{a}"].rank(ascending=False, method="first")
    for K in CALLS:
        on = Q[Q[f"rank{a}"] <= K]
        for T, g in Q.groupby("tm"):
            o = on[on["tm"] == T]; lv = g[g.y == 1]
            per_month.append({"alpha": a, "calls": K, "month": ym_of(T), "precision": o.y.mean() if len(o) else np.nan,
                              "recall_clients": o.y.sum() / max(len(lv), 1),
                              "recall_dollars": o.loc[o.y == 1, "capped"].sum() / max(lv.capped.sum(), 1)})
        fa = on[on.y == 1].sort_values("tm").drop_duplicates("cust_pwr_id")
        first_alert[(a, K)] = fa
PM_ = pd.DataFrame(per_month)
# every departing client once; its pool value is the most any list could have been credited with (max capped over its months)
pool = (Q[Q.y == 1].sort_values("tm").groupby("cust_pwr_id")
          .agg(ab_t=("ab_t", "first"), exit_type=("exit_type", "first"), capped=("capped", "max"),
               bal_norm=("bal_norm", "first"), tm=("tm", "first")).reset_index())
POOL_USD = pool.capped.sum()
yr = 12 / R["n_test"]
rows = []
for (a, K), fa in first_alert.items():
    rows.append({"alpha": a, "calls": K, "clients_reached": len(fa), "share_of_departing": len(fa) / max(len(pool), 1),
                 "dollars_reached_year": fa.capped.sum() * yr, "share_of_dollars": fa.capped.sum() / max(POOL_USD, 1),
                 **{f"save_{int(s * 100)}pct_year": fa.capped.sum() * yr * s for s in SAVE_RATES}})
VAL = pd.DataFrame(rows)
ACC = (PM_.groupby(["alpha", "calls"]).agg(precision_mean=("precision", "mean"), precision_min=("precision", "min"),
                                           precision_max=("precision", "max"), recall_clients=("recall_clients", "mean"),
                                           recall_dollars=("recall_dollars", "mean")).reset_index())
# bootstrap over departing clients for the headline cell
fa0 = first_alert[(A_MAIN, K_MAIN)]
hitcap = pool.cust_pwr_id.map(fa0.set_index("cust_pwr_id").capped).fillna(0.0).to_numpy()
reached = pool
bs = []
for _ in range(1000):
    i = rng.integers(0, len(reached), len(reached)); bs.append(hitcap[i].sum() * yr)
R.update(pool_year=POOL_USD * yr, n_departing=len(pool), val=VAL, acc=ACC, per_month=PM_,
         main_reached=fa0.capped.sum() * yr, main_ci=(float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))),
         main_clients=len(fa0), main_share=fa0.capped.sum() / max(POOL_USD, 1))
_pm = PM_[(PM_.alpha == A_MAIN) & (PM_.calls == K_MAIN)]
R["prec_ratio"] = float(_pm.precision.max() / _pm.precision.min()) if _pm.precision.min() > 0 else np.inf
B.kv([("departing clients with money still here, test months", f"{len(pool):,} · capped ${POOL_USD * yr:,.0f} per year"),
      (f"reached at {K_MAIN:,} calls, α = {A_MAIN}", f"{len(fa0):,} clients · ${R['main_reached']:,.0f} per year "
       f"(95% {R['main_ci'][0]:,.0f} – {R['main_ci'][1]:,.0f})"),
      ("share of departing dollars reached", f"{R['main_share']:.1%}"),
      ("best ÷ worst month precision", f"{R['prec_ratio']:.2f}×")], "7b · Headline")
B.table(ACC, "7c · Accuracy against calls per month (mean over test months; min/max show the spread)",
        pct=("precision_mean", "precision_min", "precision_max", "recall_clients", "recall_dollars"), save="m1_7c_accuracy")
B.table(VAL, "7d · Dollars reached and potential savings against calls (per year, no call cost)",
        pct=("share_of_departing", "share_of_dollars"),
        money=tuple(["dollars_reached_year"] + [f"save_{int(s * 100)}pct_year" for s in SAVE_RATES]), save="m1_7d_value")
B.table(PM_[PM_.calls == K_MAIN].pivot(index="month", columns="alpha", values="precision").reset_index()
        .rename(columns={a: f"precision α={a}" for a in ALPHAS}).merge(FO[["test_month", "auc_all"]], left_on="month",
                                                                      right_on="test_month").drop(columns="test_month"),
        f"7e · Accuracy by test month at {K_MAIN:,} calls", pct=tuple(f"precision α={a}" for a in ALPHAS), save="m1_7e_by_month")

# ── 7f · calibration (pooled test rows, no sampling) ─────────────────
SC["bin"] = pd.qcut(SC.p.rank(method="first"), 10, labels=False)
CAL = SC.groupby("bin").agg(mean_predicted=("p", "mean"), actual=("y", "mean"), rows=("y", "size")).reset_index()
CAL["ratio"] = CAL.actual / CAL.mean_predicted
R.update(cal=CAL)
B.table(CAL, "7f · Calibration: predicted vs actual by decile of score", pct=("mean_predicted", "actual"))

# ── charts ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for a in ALPHAS:
    d = ACC[ACC.alpha == a]
    ax[0].plot(d.calls, d.precision_mean, marker="o", label=f"α={a}")
    ax[1].plot(d.calls, d.recall_dollars, marker="o", label=f"α={a}")
    v = VAL[VAL.alpha == a]; ax[2].plot(v.calls, v.dollars_reached_year / 1e6, marker="o", label=f"α={a}")
for a_, t_ in zip(ax, ["Precision", "Share of departing dollars reached (per month)", "Dollars reached per year ($m)"]):
    a_.set_title(t_); a_.set_xlabel("calls per month"); a_.legend()
ax[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); ax[1].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
savefig(fig, "m1_7_accuracy_vs_calls")
fig, ax = plt.subplots(1, len(SAVE_RATES), figsize=(4 * len(SAVE_RATES), 3.6), sharey=False)
for i, s in enumerate(SAVE_RATES):
    for a in ALPHAS:
        v = VAL[VAL.alpha == a]; ax[i].plot(v.calls, v[f"save_{int(s * 100)}pct_year"] / 1e6, marker="o", label=f"α={a}")
    ax[i].set_title(f"Savings per year at {s:.0%} save rate ($m)"); ax[i].set_xlabel("calls per month"); ax[i].legend()
savefig(fig, "m1_7_savings_vs_calls")
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for a in ALPHAS:
    d = PM_[(PM_.alpha == a) & (PM_.calls == K_MAIN)]; ax[0].plot(d.month, d.precision, marker="o", label=f"α={a}")
ax[0].set_title(f"Precision by test month, {K_MAIN:,} calls"); ax[0].legend(); ax[0].tick_params(axis="x", rotation=90)
ax[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax[1].plot(FO.test_month, FO.auc_all, marker="o", label="all at risk"); ax[1].plot(FO.test_month, FO.auc_money_here, marker="s", label="money still here")
ax[1].set_title("AUC by test month"); ax[1].legend(); ax[1].tick_params(axis="x", rotation=90)
savefig(fig, "m1_7_by_month")
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.plot(CAL.mean_predicted, CAL.actual, marker="o"); mx = max(CAL.mean_predicted.max(), CAL.actual.max())
ax.plot([0, mx], [0, mx], color="grey", lw=0.8); ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title("Calibration by decile")
savefig(fig, "m1_7_calibration")
B.show(t0)

## §8 When signals fire; leavers vs stayers

**Event time (describes behaviour, uses the exit date):** leavers are aligned on the month their money left;
stayers get a pseudo-exit month drawn from their own live months by a fixed hash, so both sides cover the same
calendar. A signal *fires* for a client-month when the value is beyond the stayers' 10th or 90th percentile; its
lift is the leavers' firing rate over the stayers'. **Onset** is the earliest month from which the lift stays
≥ 1.5 every month up to the exit. **Model view:** the average contribution of each feature to leavers' scores
minus stayers', month by month. **Queue time** (what an RM experiences): how many months before the money left a
client first reached the list.

In [ ]:
# =====================================================================
# 8 · WHEN SIGNALS FIRE · LEAVERS VS STAYERS                       [OUTPUT BLOCK 8]
# =====================================================================
t0 = time.time(); B = Block("§8 · When signals fire")
BEHAV = [f for f in ["dep_bal_vs_norm", "dep_d6", "ntxn_all_out", "ntxn_all_in", "amt_all_out", "amt_all_in", "n_fi_ach_out",
                     "n_fi_out", "ncpty_all_out", "ncpty_all_in", "active_days_out", "rec_broken_share_out",
                     "selfpay_share_out", "net_flow_all"] if f in FEATS][:12]
TIMING = [c for c in IMP.feature if not c.startswith("miss·")][:N_TIMING]
PULL = list(dict.fromkeys(TIMING + BEHAV))
ce = FEAT.groupBy("cust_pwr_id").agg(F.first("ab_t").alias("ab_t"), F.first("last_live_t").alias("llt"))
span_pe = (LAST_LABEL_T + H) - 12 + 1
lv_ = ce.filter(F.col("ab_t").between(MIN_HIST + 2, LAST_LABEL_T + H)).withColumn("e", F.col("ab_t")).withColumn("grp", F.lit("leaver"))
st_ = (ce.filter(F.col("ab_t").isNull() & (F.col("llt") == END_T))
         .withColumn("e", F.lit(12) + F.expr(f"pmod(hash(cust_pwr_id), {span_pe})")).withColumn("grp", F.lit("stayer")))
def cap(df):
    n = df.count(); fr = min(1.0, MAX_EVENT_CUST / max(n, 1))
    return df.filter(F.expr(f"pmod(hash(cust_pwr_id, {SEED + 1}), 1000000)") < int(fr * 1_000_000))
evc = cap(lv_).unionByName(cap(st_)).select("cust_pwr_id", "e", "grp")
EV = topd(FEAT.join(F.broadcast(evc), "cust_pwr_id").withColumn("rel", F.col("t") - F.col("e"))
              .filter(F.col("rel").between(-EVENT_PRE, -1) & (F.col("live") == 1))
              .select("cust_pwr_id", "grp", "rel", *[F.col(f).cast("double").alias(f) for f in PULL]))
st_rows = EV[EV.grp == "stayer"]; lv_rows = EV[EV.grp == "leaver"]
Q10 = st_rows[TIMING].quantile(0.10); Q90 = st_rows[TIMING].quantile(0.90)
rels = list(range(-EVENT_PRE, 0)); tim, LIFT = [], {}
for f in TIMING:
    x = EV[f]; lo = (x < Q10[f]) & x.notna(); hi = (x > Q90[f]) & x.notna()
    r_lo = lo.groupby([EV.grp, EV.rel]).mean(); r_hi = hi.groupby([EV.grp, EV.rel]).mean()
    L_lo = (r_lo["leaver"] / r_lo["stayer"]).reindex(rels); L_hi = (r_hi["leaver"] / r_hi["stayer"]).reindex(rels)
    side, Lf = ("falls", L_lo) if L_lo.loc[-3:-1].mean() >= L_hi.loc[-3:-1].mean() else ("rises", L_hi)
    on = None
    for r_ in range(-1, -EVENT_PRE - 1, -1):
        if pd.notna(Lf.get(r_)) and Lf[r_] >= LIFT_BAR: on = r_
        else: break
    LIFT[f] = Lf
    tim.append({"feature": f, "label": LABEL[f], "set": FAMOF[f], "leavers": side, "onset_month": on,
                **{f"lift_{-k}m": Lf.get(k) for k in (-9, -6, -3, -1)}, "coverage": float(lv_rows[f].notna().mean())})
TIM = pd.DataFrame(tim).sort_values(["onset_month", "lift_1m"], ascending=[True, False], na_position="last").reset_index(drop=True)
R.update(tim=TIM, n_pay_onset6=int(((TIM.onset_month <= -6) & ~TIM.feature.str.startswith("dep_")).sum()),
         n_ev_leavers=int(lv_rows.cust_pwr_id.nunique()), n_ev_stayers=int(st_rows.cust_pwr_id.nunique()))
B.table(TIM, f"8a · Event time: when each of the top {len(TIMING)} features fires (lift = leavers ÷ stayers beyond the stayers' 10th/90th percentile)",
        pct=("coverage",), save="m1_8a_onset")
fig, ax = plt.subplots(figsize=(12, 0.32 * len(TIM) + 1.5))
M_ = np.array([LIFT[f].values for f in TIM.feature], dtype=float)
im = ax.imshow(np.clip(M_, 0, 5), aspect="auto", cmap="Reds", vmin=1, vmax=5)
ax.set_yticks(range(len(TIM))); ax.set_yticklabels([f"{l[:60]} ({d})" for l, d in zip(TIM.label, TIM.leavers)], fontsize=8)
ax.set_xticks(range(len(rels))); ax.set_xticklabels(rels); ax.set_xlabel("months before the money left")
ax.set_title("Lift of each signal over stayers (1 = no difference, capped at 5)"); fig.colorbar(im, ax=ax, shrink=0.6)
savefig(fig, "m1_8a_onset_heatmap")

# ── model view: contribution to leavers' scores minus stayers', by month ──
P_, b_ = MODEL["P"], MODEL["b"][1:]
MV = {}
for f in [c for c in TIMING][:15]:
    j = FEATS.index(f)
    z = (np.clip(EV[f].to_numpy(float), P_["lo"][j], P_["hi"][j]) - P_["mu"][j]) / P_["sd"][j]
    c = b_[j] * np.nan_to_num(z)
    s = pd.Series(c).groupby([EV.grp.values, EV.rel.values]).mean()
    MV[LABEL[f][:60]] = (s["leaver"] - s["stayer"]).reindex(rels)
MVT = pd.DataFrame(MV).T; MVT.columns = [str(c) for c in rels]
MVT = MVT.reset_index().rename(columns={"index": "feature"})
lead6 = MVT.set_index("feature")["-6"].sort_values(ascending=False).head(5).index.tolist()
lead1 = MVT.set_index("feature")["-1"].sort_values(ascending=False).head(5).index.tolist()
R.update(model_view=MVT, lead6=lead6, lead1=lead1)
B.table(MVT, "8b · Model view: contribution to leavers' score minus stayers' (log-odds), by month before the money left",
        save="m1_8b_model_view")
B.kv([("largest contributions 6 months out", " · ".join(lead6)), ("largest contributions 1 month out", " · ".join(lead1))],
     "8b · Early vs late drivers")

# ── queue time ───────────────────────────────────────────────────────
fa0 = first_alert[(A_MAIN, K_MAIN)].copy()
fa0["lead"] = fa0.ab_t - fa0["tm"]
LEAD = fa0.groupby("lead").agg(clients=("cust_pwr_id", "size"), capped=("capped", "sum")).reset_index()
LEAD["share_clients"] = LEAD.clients / LEAD.clients.sum(); LEAD["share_dollars"] = LEAD.capped / LEAD.capped.sum()
fa0["reason_set"] = fa0.r1.map(FAMOF)
RS = fa0.groupby("reason_set").agg(clients=("cust_pwr_id", "size")).reset_index().sort_values("clients", ascending=False)
RS["share"] = RS.clients / RS.clients.sum()
R.update(lead=LEAD, lead3=float((fa0.lead >= 3).mean()) if len(fa0) else np.nan,
         lead3_usd=float(fa0.loc[fa0.lead >= 3, "capped"].sum() / max(fa0.capped.sum(), 1)), reason_sets=RS)
B.table(LEAD, f"8c · Queue time: months between the first alert and the money leaving ({K_MAIN:,} calls, α = {A_MAIN})",
        pct=("share_clients", "share_dollars"), money=("capped",), save="m1_8c_lead")
B.table(RS, "8c · Top reason at the first alert, by feature set", pct=("share",))

# ── leavers vs stayers ───────────────────────────────────────────────
BV = EV.groupby(["grp", "rel"])[BEHAV].median()
fig, axs = plt.subplots(int(np.ceil(len(BEHAV) / 4)), 4, figsize=(16, 3.1 * int(np.ceil(len(BEHAV) / 4))))
for a_, f in zip(np.ravel(axs), BEHAV):
    for g_, col in (("leaver", "#c00000"), ("stayer", "#1f4e79")):
        a_.plot(rels, BV.loc[g_, f].reindex(rels), marker="o", ms=3, color=col, label=g_)
    a_.set_title(LABEL[f][:48], fontsize=9); a_.tick_params(labelsize=8)
for a_ in np.ravel(axs)[len(BEHAV):]: a_.axis("off")
np.ravel(axs)[0].legend(fontsize=8); fig.suptitle("Median by month before the money left (model units: log1p for dollars and counts)")
savefig(fig, "m1_8d_leavers_vs_stayers")
LS = []
for f in BEHAV:
    for k in (-12, -6, -3, -1):
        LS.append({"feature": LABEL[f], "month": k, "leavers": BV.loc["leaver", f].get(k), "stayers": BV.loc["stayer", f].get(k)})
LS = pd.DataFrame(LS).pivot(index="feature", columns="month", values=["leavers", "stayers"])
LS.columns = [f"{a} {b}" for a, b in LS.columns]; LS = LS.reset_index()
R.update(ls=LS, behav=BEHAV)
B.table(LS, "8d · Leavers vs stayers: medians 12, 6, 3 and 1 months before", save="m1_8d_leavers_vs_stayers")
B.kv([("leavers / stayers traced", f"{R['n_ev_leavers']:,} / {R['n_ev_stayers']:,}"),
      ("payment features firing 6+ months out", R["n_pay_onset6"]),
      (f"departing clients first alerted 3+ months before the money left", f"{R['lead3']:.1%} of clients, {R['lead3_usd']:.1%} of dollars")],
     "8e · Summary")
B.show(t0)

## §9 Example clients for RM managers

Chosen **by rule** from the test months, at 1,000 calls and α = 0.5, so nobody hand-picked a flattering case:
an early catch, a case where payments moved before the balance, a client who drained without closing, a late
catch, the most confident false alarm, and the largest miss. Each shows the balance against normal, the months
on the list, the reasons the model gave, the event-time signals that fired, and the window an RM had to act.
Ids are one-way hashes; full ids go only to `m1_9_examples_FULL_IDS.csv`. Review each with its RM before
showing it more widely.

In [ ]:
# =====================================================================
# 9 · EXAMPLE CLIENTS FOR RM MANAGERS                               [OUTPUT BLOCK 9]
# =====================================================================
t0 = time.time(); B = Block("§9 · Example clients")
def play(feat):
    f = feat.replace("miss·", "")
    if re.search(r"(n_fi|^fi_|_fi_|selfpay|top_fi|^share_|n_rails)", f):
        return ("Competitive threat: money appears to be moving to another bank. Compare pricing, earnings credit "
                "and services with the likely competitor; offer a retention package.")
    if f.startswith("dep_"):
        return ("Balance drawdown: deposits are falling ahead of the payments. Ask about cash plans; offer sweep, "
                "money-market or liquidity options to keep the operating balance here.")
    return ("Business activity is shrinking: fewer payments and trading partners. Check the client's business; "
            "offer credit, working-capital or payment-product support.")

rk = f"rank{A_MAIN}"
Qx = Q[["cust_pwr_id", "tm", rk, "p", "bal", "bal_norm", "capped", "ab_t", "exit_type", "y", "r1", "r2", "r3", "c1", "c2", "c3"]]
# first month each departing client was on the list before its money left (any month, not only the last six)
lst = (Qx[(Qx[rk] <= K_MAIN) & (Qx.tm < Qx.ab_t)].sort_values("tm").drop_duplicates("cust_pwr_id")
         [["cust_pwr_id", "tm", "bal", "bal_norm", "r1"]]
         .rename(columns={"tm": "T_alert", "bal": "bal_at_alert", "bal_norm": "norm_at_alert"}))
dp = pool[["cust_pwr_id", "ab_t", "exit_type", "capped", "bal_norm"]].merge(lst, on="cust_pwr_id", how="left")
dp["lead"] = dp.ab_t - dp.T_alert
dp["pct_at_alert"] = dp.bal_at_alert / dp.norm_at_alert
hit = dp[dp.T_alert.notna()]
chosen, EXS = set(), []
def pick(df, kind, sort="bal_norm"):
    d = df[~df.cust_pwr_id.isin(chosen)].sort_values(sort, ascending=False)
    if len(d):
        chosen.add(d.cust_pwr_id.iloc[0]); EXS.append((kind, d.cust_pwr_id.iloc[0]))
pick(hit[hit.lead >= 4], "Early catch")
pick(hit[(hit.pct_at_alert >= 0.9) & hit.r1.map(lambda r: isinstance(r, str) and not r.startswith("dep_"))],
     "Payments moved before the balance")
pick(hit[hit.exit_type == "balance exit"], "Drained without closing")
pick(hit[hit.lead <= 2], "Late catch")
never = set(SC.loc[SC.ab_t.isna(), "cust_pwr_id"]) - set(SC.loc[SC.y == 1, "cust_pwr_id"])
fal = Qx[(Qx[rk] <= K_MAIN) & Qx.cust_pwr_id.isin(never)].sort_values("p", ascending=False)
if len(fal): chosen.add(fal.cust_pwr_id.iloc[0]); EXS.append(("False alarm (most confident wrong call)", fal.cust_pwr_id.iloc[0]))
pick(dp[dp.T_alert.isna()], "Missed", sort="capped")

ids = [c for _, c in EXS]
PAN = topd(FEAT.filter(F.col("cust_pwr_id").isin(ids))
               .select("cust_pwr_id", "t", "bal", "bal_norm", "live", "ab_t", *[F.col(f).cast("double").alias(f) for f in TIMING]))
tim_on = TIM.dropna(subset=["onset_month"])
EXOUT, full_ids = [], []
for n, (kind, c) in enumerate(EXS, 1):
    pan = PAN[PAN.cust_pwr_id == c].sort_values("t"); sc = Qx[Qx.cust_pwr_id == c].sort_values("tm")
    e = pan.ab_t.dropna().iloc[0] if pan.ab_t.notna().any() else None
    lo_t = int(min(sc["tm"].min() if len(sc) else pan.t.max(), (e - 12) if e is not None else 10 ** 6))
    hi_t = int(e + 1) if e is not None else int(sc["tm"].max() + 1 if len(sc) else pan.t.max())
    pan = pan[(pan.t >= max(lo_t, 0)) & (pan.t <= hi_t)]
    ref = float(sc.bal_norm.iloc[0]) if len(sc) else float(pan.bal_norm.dropna().median())
    # event-time signals that fired for this client (same thresholds and directions as §8a)
    fired = []
    for _, s in tim_on.iterrows():
        f = s.feature; th = Q10[f] if s.leavers == "falls" else Q90[f]
        v = pan.set_index("t")[f]
        hitm = v[(v < th) if s.leavers == "falls" else (v > th)].index
        if len(hitm) and e is not None and (hitm < e).any():
            fired.append((int(min(hitm[hitm < e])) - int(e), s.label))
    fired = sorted(fired)[:5]
    alerts = sc[sc[rk] <= K_MAIN]
    rows = []
    for _, r in pan.iterrows():
        s_ = sc[sc["tm"] == r.t]
        rows.append({"month": ym_of(r.t), "vs_exit": (int(r.t - e) if e is not None else None),
                     "balance_pct_normal": r.bal / ref if ref else np.nan,
                     "score": float(s_.p.iloc[0]) if len(s_) else np.nan,
                     "rank": int(s_[rk].iloc[0]) if len(s_) else None,
                     "on_list": bool(len(s_) and s_[rk].iloc[0] <= K_MAIN),
                     "reasons": " · ".join(LABEL.get(x, x)[:60] for x in s_[["r1", "r2", "r3"]].iloc[0]) if len(s_) else ""})
    TAB = pd.DataFrame(rows)
    first = alerts.iloc[0] if len(alerts) else None
    if kind.startswith("False"):
        text = (f"On the list in {len(alerts)} test month(s) with a score up to {sc.p.max():.1%}, and stayed. "
                f"Reasons given: {TAB.loc[TAB.on_list, 'reasons'].iloc[0] if TAB.on_list.any() else '—'}. "
                f"This is what a wrong call looks like; at {K_MAIN:,} calls about "
                f"{1 - float(ACC[(ACC.alpha == A_MAIN) & (ACC.calls == K_MAIN)].precision_mean.iloc[0]):.0%} of calls are like it.")
    elif first is None:
        best = sc.sort_values(rk).iloc[0] if len(sc) else None
        text = (f"The money left in {ym_of(e)} ({dp.loc[dp.cust_pwr_id == c, 'exit_type'].iloc[0]}) "
                f"and the client never reached the top {K_MAIN:,}" +
                (f"; best rank {int(best[rk]):,} in {ym_of(best['tm'])}, reasons: {' · '.join(LABEL.get(x, x)[:50] for x in best[['r1','r2','r3']])}."
                 if best is not None else ".") +
                (" Signals that did fire: " + "; ".join(f"{l[:55]} (from {m})" for m, l in fired) + "." if fired else ""))
    else:
        pct = first.bal / first.bal_norm if first.bal_norm else np.nan
        text = (f"First on the list in {ym_of(first['tm'])}, {int(e - first['tm'])} month(s) before the money left "
                f"({dp.loc[dp.cust_pwr_id == c, 'exit_type'].iloc[0]}, {ym_of(e)}). The balance was then {pct:.0%} of its "
                f"${first.bal_norm:,.0f} normal. Reasons: {' · '.join(LABEL.get(x, x)[:60] for x in first[['r1','r2','r3']])}. " +
                ("Signals that fired: " + "; ".join(f"{l[:55]} (from {m})" for m, l in fired) + ". " if fired else "") +
                f"How an RM could have intervened: {play(first.r1)}")
    fig, ax = plt.subplots(figsize=(10, 3.4))
    ax.plot([ym_of(t) for t in pan.t], pan.bal / ref, marker="o", color="#1f4e79", label="balance, % of normal")
    for _, a_ in alerts.iterrows():
        if a_["tm"] in set(pan.t): ax.axvline(ym_of(a_["tm"]), color="#c55a11", alpha=0.35, lw=9)
    if e is not None and e in set(pan.t): ax.axvline(ym_of(e), color="#c00000", ls="--", label="money left")
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); ax.tick_params(axis="x", rotation=90)
    ax.set_title(f"Example {n} · {kind} · client {cid(c)} (orange = on the list)"); ax.legend(loc="lower left", fontsize=8)
    savefig(fig, f"m1_9_example_{n}")
    EXOUT.append(dict(n=n, kind=kind, client=cid(c), normal=ref, text=text, table=TAB, chart=f"m1_9_example_{n}"))
    full_ids.append({"example": n, "kind": kind, "display_id": cid(c), "cust_pwr_id": c})
    B.note(f"<b>Example {n} · {kind} · client {cid(c)}</b> — {_html.escape(text)}")
    B.table(TAB, f"Example {n}: month by month", pct=("balance_pct_normal", "score"))
pd.DataFrame(full_ids).to_csv(OUT / "m1_9_examples_FULL_IDS.csv", index=False)
R.update(examples=EXOUT)
if len(EXS) < 6: B.check("examples", "INFO", f"only {len(EXS)} of 6 rules found a client")
B.show(t0)

## §10 Prediction scorecard and the HTML report

The report has the same sections as the v1–v10d programme report. Section 1 ("What we found") is left as a
placeholder with the headline numbers filled in: the findings are written after the results have been read.

In [ ]:
# =====================================================================
# 10 · PREDICTIONS · HTML REPORT                                    [OUTPUT BLOCK 10]
# =====================================================================
t0 = time.time(); B = Block("§10 · Predictions and report")
_v = lambda a, K: float(VAL[(VAL.alpha == a) & (VAL.calls == K)].dollars_reached_year.iloc[0])
PRED = [
    ("P1", "Pooled test AUC ≥ 0.80", 0.60, lambda: R["auc_pooled"] >= 0.80, f"{R['auc_pooled']:.3f}"),
    ("P2", "Largest contribution set is a payment set", 0.40, lambda: FAMI.set.iloc[0].startswith("payment"), FAMI.set.iloc[0]),
    ("P3", "A bank-count feature (n_fi*) in the top 10", 0.55,
     lambda: any(f.split("__")[0].replace("miss·", "").startswith("n_fi") for f in IMP.feature.head(10)),
     ", ".join(f for f in IMP.feature.head(10) if "n_fi" in f) or "none"),
    ("P4", f"Best ÷ worst month precision ≤ 2× ({K_MAIN:,} calls, α={A_MAIN})", 0.50, lambda: R["prec_ratio"] <= 2, f"{R['prec_ratio']:.2f}×"),
    ("P5", f"α=0.5 reaches more dollars than α=0 at {K_MAIN:,} calls", 0.80, lambda: _v(0.5, K_MAIN) > _v(0.0, K_MAIN),
     f"${_v(0.5, K_MAIN):,.0f} vs ${_v(0.0, K_MAIN):,.0f}"),
    ("P6", "≥ 30% of departing clients first alerted 3+ months ahead", 0.55, lambda: R["lead3"] >= 0.30, f"{R['lead3']:.1%}"),
    ("P7", "≥ 3 payment features fire 6+ months ahead", 0.50, lambda: R["n_pay_onset6"] >= 3, f"{R['n_pay_onset6']}")]
rows = []
for pid, txt, conf, fn, obs in PRED:
    try: hit = bool(fn())
    except Exception: hit = None
    rows.append({"id": pid, "prediction": txt, "confidence": conf, "observed": obs,
                 "result": "HELD" if hit else "FAILED" if hit is False else "n/a",
                 "brier": (conf - float(hit)) ** 2 if hit is not None else np.nan})
PS = pd.DataFrame(rows); R["preds"] = PS
B.table(PS, "10a · Pre-registered predictions", pct=("confidence",), save="m1_10a_predictions")

# ── report ───────────────────────────────────────────────────────────
def img(name):
    return f"<img src='data:image/png;base64,{base64.b64encode(open(CHARTS[name], 'rb').read()).decode()}' style='max-width:100%'>"
def tab(df, pct=(), money=(), rows=60):
    return fmt_df(df.head(rows), pct, money).to_html(index=False, border=0, classes="t", escape=True)
usd = lambda v: f"${v / 1e6:,.1f}m" if abs(v) >= 1e6 else f"${v:,.0f}"
S = []
S.append(f"""<h1>Deposit attrition early warning — v13, one model on deposit and payment features</h1>
<p class='sub'>Payment Knowledge Graph (PKG) · Treasury Management · Data Science · test months {R['test_months'][0]} … {R['test_months'][-1]}</p>""")
S.append(f"""<section><h2>1 · What we found</h2>
<div class='box warn'><b>To be written after the results are read.</b> The headline numbers below are filled in automatically.</div>
{tab(pd.DataFrame([
 ["clients whose money left, per year", f"{R['exits_per_year']:,.0f}"],
 ["their normal balance, per year", usd(R['at_risk_year'])],
 ["pooled test AUC (all at risk / money still here)", f"{R['auc_pooled']:.3f} / {R['auc_pooled_msh']:.3f}"],
 [f"dollars reached per year at {K_MAIN:,} calls, α = {A_MAIN}", f"{usd(R['main_reached'])} (95% {usd(R['main_ci'][0])} – {usd(R['main_ci'][1])})"],
 ["share of departing dollars reached", f"{R['main_share']:.1%}"],
 ["first alerted 3+ months before the money left", f"{R['lead3']:.1%} of clients"],
 ["savings per year at a 5% save rate", usd(R['main_reached'] * 0.05)]], columns=["measure", "value"]))}</section>""")
S.append("""<section><h2>2 · The question</h2><p>Which Treasury Management deposit clients will move their money out in the
next six months, early enough for a relationship manager to act, and how much of that money a fixed number of calls
can reach? This version rebuilds the model from scratch on the new PKG customer metrics and the deposit data, with a
single simple model on all features.</p></section>""")
CK = pd.DataFrame(CHECKS)
S.append(f"""<section><h2>3 · Data and preparation</h2>
<h3>Sources</h3>{tab(pd.DataFrame([[DEP_TABLE, "daily balances and status per account × leg", f"{DATE_START[:7]} … {R['dep_end']}"],
   [MET_TABLE, f"{R['n_metrics']} monthly payment metrics per customer", f"{DATE_START[:7]} … {R['dep_end']}"]],
   columns=["table", "content", "months"]))}
<h3>Pipeline</h3><ol><li>Deposit legs summed per account-day; balances rolled to customer-day, then month (average, minimum,
maximum, month-end). Status codes from both vocabularies (HOGAN numeric, sweep letters) mapped to live / closed.</li>
<li>Every customer gets a complete month grid, so a month with no rows reads as non-live with zero balance.</li>
<li>Payment metrics joined on the power id (deposit <code>cust_pwr_id</code> = metrics <code>customer_pwr_id</code>,
{R['key_match']:.1%} of deposit customers found). Where one power id has several payment entities, the largest one that month is used.</li>
<li>Each metric becomes a level, a change vs the client's months −6…−4 and a change vs its last 3 months; dollars and
counts are on a log scale, so a change is a log ratio and a halving weighs as much as a doubling.</li></ol>
<h3>Silent months and missing values</h3><p>A month with no payment row after the client's first row is <b>zero
activity</b> (dollars and counts 0), not missing. Shares with nothing to divide by stay empty. Churn-state metrics are
empty for the first 6 months of 2024 while their lookback is too short. Every empty value is set to the training mean
after standardisation, and one flag per distinct missing pattern ({R['n_flags']} flags) lets the model use the
absence itself. {R['pay_seen']:.1%} of live client-months have payment history ({R['pay_seen_usd']:.1%} of balance);
{R['pay_silent']:.1%} are silent.</p>
<h3>Checks</h3>{tab(CK)}</section>""")
S.append(f"""<section><h2>4 · Definitions</h2>
<h3>The target, and why</h3>
<div class='box'><b>The money leaves within the next 6 months</b>: the earlier of a <b>full exit</b> (every deposit account
non-live, confirmed by {CONFIRM_M} observed non-live months) and a <b>balance exit</b> (balance below {BAL_EXIT_FRAC:.0%} of the
client's normal balance for {BAL_EXIT_HOLD} consecutive months while accounts stay open).</div>
<ul><li><b>It is what costs money.</b> A client who moves its balance and leaves a shell account open is as lost as one
who closes; a closure-only target never flags it.</li>
<li><b>It is earlier.</b> A balance exit usually comes before the formal closure, so the model is trained to recognise
the move, not the paperwork.</li>
<li><b>Full exits are absorbing:</b> {R['p_return']:.2%} of clients observed non-live for {CONFIRM_M}+ months came back,
so a confirmed full exit is a departure, not a pause.</li>
<li><b>Six months</b> is long enough for an RM to act and short enough that the signals are still about this client's
decision. Months whose next six months are not fully observed ({R['last_label']} onwards) are not labelled.</li></ul>
<h3>Terms</h3>{tab(pd.DataFrame([
 ["normal balance", f"median balance over up to {NORM_WIN} prior live months (needs {NORM_MIN_OBS})"],
 ["money still here", f"live, normal balance ≥ ${MSH_FLOOR:,.0f}, current month ≥ {MSH_FRAC:.0%} of normal; only these clients are called"],
 ["capped credit", "what a save is credited with: the lesser of current and normal balance, so a one-off inflow never counts"],
 ["α", "score = probability × normal balance^α; 0 ranks by probability, 1 by expected dollars"],
 ["dollars reached", "capped balance of departing clients at their first alert, each client counted once, annualised"],
 ["savings", "dollars reached × an assumed save rate (1%, 5%, 10%, 20%); there is no call cost"]], columns=["term", "meaning"]))}</section>""")
S.append(f"""<section><h2>5 · Training and test sets</h2><p>Rolling monthly folds. For each test month the model is
fitted only on months whose outcome was already known then (origins ≤ test month − 6) and scores every client at
risk that month. Training keeps every leaving client-month and {NEG_RATE:.0%} of the others (fixed hash), weighted
{1 / NEG_RATE:.0f}× so the probabilities stay on the population scale.</p>
{tab(R['folds'], pct=('mean_p', 'actual_rate'))}</section>""")
S.append(f"""<section><h2>6 · Feature sets</h2><p>{R['n_feats']} features plus {R['n_flags']} missing-pattern flags. No identity
columns. Deposit subsections come from the deposit data; payment sets are the metric dictionary's families.</p>
{tab(R['feature_sets'], pct=('median_coverage',))}</section>""")
S.append(f"""<section><h2>7 · Methods</h2><h3>Model</h3><p>Ridge logistic regression (L2 = {L2}) on standardised features
clipped at the {CLIP_Q[0]:.1%} / {CLIP_Q[1]:.1%} quantiles, fitted by damped Newton so every step improves the
objective. Each score splits exactly into per-feature contributions, which give the reason codes.</p>
<h3>Validation without leakage</h3><p>All features look backwards; labels need six confirmed future months; each fold
trains only on outcomes known before its test month; the test month is scored in full, with no sampling.</p>
<h3>From a score to dollars</h3><p>Each month the clients with money still here are ranked by p × normal^α and the top
K are called; a departing client is credited once, at its first alert, with its capped balance.</p>
<h3>Uncertainty</h3><p>Accuracy is shown per test month, so no single month carries the result; the headline dollars
carry a 95% interval from resampling departing clients.</p></section>""")
S.append(f"""<section><h2>8 · Results</h2>
<h3>8.1 How much is at risk</h3><p>{R['exits_per_year']:,.0f} clients a year move their money out, with {usd(R['at_risk_year'])} of normal
balance; {R['msh_m3']:.0%} still had their money here three months before. The largest single client is
{R['top_client_share']:.1%} of the total.</p>{tab(R['at_risk_tab'], money=('normal_balance', 'per_year'))}
<h3>8.2 Which features predict</h3>{img('m1_6_importance')}
{tab(R['imp'].head(25), pct=('model_share', 'coverage'))}<h4>By feature set</h4>{tab(R['fam_imp'], pct=('model_share',))}
<h4>Strongest on their own</h4>{tab(R['alone'][['feature', 'label', 'set', 'alone_auc', 'leavers_sit', 'coverage']], pct=('coverage',))}
<h3>8.3 Accuracy against calls</h3>{img('m1_7_accuracy_vs_calls')}
{tab(R['acc'], pct=('precision_mean', 'precision_min', 'precision_max', 'recall_clients', 'recall_dollars'))}
<h3>8.4 Accuracy by test month</h3>{img('m1_7_by_month')}
<h3>8.5 Calibration</h3>{img('m1_7_calibration')}{tab(R['cal'], pct=('mean_predicted', 'actual'))}
<h3>8.6 When signals fire</h3><p>Event time describes behaviour: it uses the exit date, which nobody has when the list
is made. Queue time is what an RM experiences.</p>{img('m1_8a_onset_heatmap')}{tab(R['tim'], pct=('coverage',))}
<h4>Model view: which features drive leavers' scores, month by month</h4>{tab(R['model_view'])}
<p>Largest 6 months out: {', '.join(R['lead6'])}. Largest 1 month out: {', '.join(R['lead1'])}.</p>
<h4>Queue time</h4>{tab(R['lead'], pct=('share_clients', 'share_dollars'), money=('capped',))}
{tab(R['reason_sets'], pct=('share',))}
<h3>8.7 Savings against calls</h3>{img('m1_7_savings_vs_calls')}
{tab(R['val'], pct=('share_of_departing', 'share_of_dollars'), money=tuple(['dollars_reached_year'] + [f'save_{int(s * 100)}pct_year' for s in SAVE_RATES]))}
<h3>8.8 Leavers and stayers</h3>{img('m1_8d_leavers_vs_stayers')}{tab(R['ls'])}
<h3>8.9 Examples for RM managers</h3><p>Chosen by rule; review each with the client's RM before wider use.</p>""" +
"".join(f"<h4>Example {e['n']} · {e['kind']} · client {e['client']}</h4><p>{_html.escape(e['text'])}</p>{img(e['chart'])}"
        f"{tab(e['table'], pct=('balance_pct_normal', 'score'))}" for e in R["examples"]) + "</section>")
S.append(f"""<section><h2>9 · Predictions written before the run</h2>{tab(R['preds'], pct=('confidence',))}</section>
<section><h2>10 · Limitations</h2><ul>
<li><b>The save rate is unmeasured.</b> Savings are dollars reached × an assumed rate; only a pilot with a holdout can measure it.</li>
<li><b>The model is linear.</b> Interactions and thresholds are only captured through the transforms; a tree model is the next step.</li>
<li><b>Payment metrics are current-identity:</b> a customer's power id is today's, applied to all months.</li>
<li><b>Few test months</b> ({R['n_test']}) because the data start in 2024 and every label needs six confirmed months.</li>
<li><b>Event-time onsets describe leavers after the fact</b>; they are not detection leads.</li></ul></section>
<section><h2>11 · Next steps</h2><ul><li>Gradient-boosted trees (XGBoost) in the same harness, judged on dollars reached.</li>
<li>Review the example clients with their RMs.</li><li>A pilot with a randomised holdout to measure the save rate.</li></ul></section>
<section><h2>Appendix · Parameters</h2>{tab(pd.DataFrame([[k, str(v)] for k, v in [("H", H), ("CONFIRM_M", CONFIRM_M),
 ("NORM_WIN / NORM_MIN_OBS", f"{NORM_WIN} / {NORM_MIN_OBS}"), ("MIN_HIST", MIN_HIST), ("money still here", f"{MSH_FRAC} × normal, ≥ ${MSH_FLOOR:,}"),
 ("balance exit", f"< {BAL_EXIT_FRAC} × normal for {BAL_EXIT_HOLD} months"), ("CHURN_WARM", CHURN_WARM), ("NEG_RATE", NEG_RATE),
 ("L2", L2), ("CLIP_Q", CLIP_Q), ("CALLS", CALLS), ("ALPHAS", ALPHAS), ("SAVE_RATES", SAVE_RATES), ("LIFT_BAR", LIFT_BAR)]],
 columns=["parameter", "value"]))}</section>""")
CSS = """body{font-family:Arial,Helvetica,sans-serif;max-width:1180px;margin:24px auto;padding:0 18px;color:#222;line-height:1.45}
h1{font-size:26px;margin-bottom:4px}.sub{color:#666;margin-top:0}h2{border-bottom:2px solid #1f4e79;padding-bottom:4px;margin-top:34px}
h3{color:#1f4e79;margin-top:22px}table.t{border-collapse:collapse;font-size:12px;margin:8px 0;display:block;overflow-x:auto}
table.t th{background:#1f4e79;color:#fff;padding:4px 8px;text-align:left}table.t td{padding:3px 8px;border-bottom:1px solid #e3e3e3}
.box{background:#eef4fa;border-left:4px solid #1f4e79;padding:10px 14px;margin:10px 0}.box.warn{background:#fff4e5;border-color:#e65100}
code{background:#f3f3f3;padding:0 3px}"""
REPORT = OUT / "PKG_Attrition_v13_m1_report.html"
REPORT.write_text(f"<!doctype html><html><head><meta charset='utf-8'><title>PKG attrition v13 · model 1</title>"
                  f"<style>{CSS}</style></head><body>{''.join(S)}"
                  f"<p style='color:#999;font-size:11px'>Internal — PNC Treasury Management, Data Science</p></body></html>", encoding="utf-8")
B.kv([("report", str(REPORT.resolve())), ("size", f"{REPORT.stat().st_size / 1e6:,.1f} MB"),
      ("CSVs and charts", str(OUT.resolve())), ("total wall time", f"{(time.time() - T0) / 60:,.1f} min")], "10b · Written")
for d_ in (X, FEAT, MF): d_.unpersist()
B.show(t0)